## GreenCart E-commerce A/B Test Analysis
**Business Objective:** Determine if the new website design (Version B) significantly improves conversion rates compared to the current design (Version A).

* **Version A:** Current design
* **Version B:** Simplified navigation, larger images

In [1]:
import numpy as np
from scipy import stats

# Data: Version A (Current)
conversions_a = 150    
visitors_a = 1000      

# Data: Version B (New)
conversions_b = 210
visitors_b = 1200

### Part 1: Current Website Performance
First, we calculate GreenCart's current conversion rate and use confidence intervals to estimate the true population rate.

In [2]:
def analyze_conversion_rate(conversions, visitors, confidence_level=0.95):
    p_hat = conversions / visitors
    print(f"Current conversion rate: {p_hat:.1%}")

    std_error = np.sqrt((p_hat * (1 - p_hat)) / visitors)
    z_critical = stats.norm.ppf((1 + confidence_level) / 2)
    margin_error = z_critical * std_error

    ci_lower = max(0, p_hat - margin_error) 
    ci_upper = min(1, p_hat + margin_error) 
    
    print(f"{confidence_level*100}% Confidence Interval: ({ci_lower:.1%}, {ci_upper:.1%})")
    
    return {'confidence_interval': (ci_lower, ci_upper)}

current_performance = analyze_conversion_rate(conversions_a, visitors_a)

Current conversion rate: 15.0%
95.0% Confidence Interval: (12.8%, 17.2%)


### Part 2: Hypothesis Testing
We see a 2.5% increase in conversions with the new design (17.5% vs 15.0%). Next, we run a two-proportion Z-test to see if this difference is statistically significant, or just random chance.

In [5]:
def compare_designs(conversions_a, visitors_a, conversions_b, visitors_b, alpha=0.05):
    p_a = conversions_a / visitors_a
    p_b = conversions_b / visitors_b
    
    # Calculate standard error and test statistic
    p_pooled = (conversions_a + conversions_b) / (visitors_a + visitors_b)
    se_diff = np.sqrt(p_pooled * (1 - p_pooled) * (1/visitors_a + 1/visitors_b))
    z_stat = (p_b - p_a) / se_diff
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    
    print(f"P-value: {p_value:.4f}")
    if p_value < alpha:
        print("Statistically significant (Reject Null)")
    else:
        print("Not statistically significant (Fail to Reject Null)")
    
    return {'difference': p_b - p_a, 'p_value': p_value}

comparison_results = compare_designs(conversions_a, visitors_a, conversions_b, visitors_b)

P-value: 0.1145
Not statistically significant (Fail to Reject Null)


### Part 3: Business Summary
Finally, we translate these statistical findings into actionable advice for the marketing team.

In [7]:
def prepare_business_summary(current_perf, comparison):
    print("\nSummary for Sarah")
    if comparison['p_value'] < 0.05:
        print(f"Impact: The new design significantly changed the conversion rate by {comparison['difference']:.1%}.")
        print("Recommendation: Implement the new design.")
    else:
        print("Impact: The observed difference is not statistically significant.")
        print("Recommendation: Do not switch yet. Continue testing to gather more data.")

prepare_business_summary(current_performance, comparison_results)


Summary for Sarah
Impact: The observed difference is not statistically significant.
Recommendation: Do not switch yet. Continue testing to gather more data.
